# NLP Getting Started Tutorial (Colab — Kaggle 다운로드 버전)

Kaggle의 [NLP Getting Started Tutorial](https://www.kaggle.com/code/philculliton/nlp-getting-started-tutorial) (by Phil Culliton) 을 **Google Colab에서 실행**하는 노트북입니다.

- 실행 시 **Kaggle API로 대회 데이터를 직접 다운로드**합니다 ([NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) 의 `train.csv`/`test.csv`).
- Kaggle API 토큰이 **노트북에 내장**되어 있어 별도 입력 없이 바로 실행됩니다.
- 모델: scikit-learn `CountVectorizer` + `RidgeClassifier` (가벼움, CPU로 충분).
- 마지막에 제출 파일 `submission.csv` 가 `/content` 에 생성됩니다.

> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.  
> (토큰 재발급/폐기: https://www.kaggle.com/settings/api)

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
# 필요한 라이브러리 설치
import sys, subprocess
for pkg in ["kaggle", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 Disaster Tweets 데이터 다운로드

In [ ]:
import os, glob, zipfile

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

api.competition_download_files("nlp-getting-started", path=WORK_DIR, quiet=False)
for zpath in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(WORK_DIR)
    os.remove(zpath)

print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## NLP Tutorial

NLP - or *Natural Language Processing* - is shorthand for a wide array of techniques designed to help machines learn from text. Natural Language Processing powers everything from chatbots to search engines, and is used in diverse tasks like sentiment analysis and machine translation.

In this tutorial we'll look at this competition's dataset, use a simple technique to process it, build a machine learning model, and submit predictions for a score!

In [ ]:
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn import feature_extraction, linear_model, model_selection, preprocessing

In [ ]:
train_df = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))

### A quick look at our data

Let's look at our data... first, an example of what is NOT a disaster tweet.

In [ ]:
train_df[train_df["target"] == 0]["text"].values[1]

And one that is:

In [ ]:
train_df[train_df["target"] == 1]["text"].values[1]

### Building vectors

The theory behind the model we'll build in this notebook is pretty simple: the words contained in each tweet are a good indicator of whether they're about a real disaster or not (this is not entirely correct, but it's a great place to start).

We'll use scikit-learn's `CountVectorizer` to count the words in each tweet and turn them into data our machine learning model can process.

Note: a `vector` is, in this context, a set of numbers that a machine learning model can work with. We'll look at one in just a second.

In [ ]:
count_vectorizer = feature_extraction.text.CountVectorizer()

## let's get counts for the first 5 tweets in the data
example_train_vectors = count_vectorizer.fit_transform(train_df["text"][0:5])

In [ ]:
## we use .todense() here because these vectors are "sparse" (only non-zero elements are kept to save space)
print(example_train_vectors[0].todense().shape)
print(example_train_vectors[0].todense())

The above tells us that:
1. There are 54 unique words (or "tokens") in the first five tweets.
2. The first tweet contains only some of those unique tokens - all of the non-zero counts above are the tokens that DO exist in the first tweet.

Now let's create vectors for all of our tweets.

In [ ]:
train_vectors = count_vectorizer.fit_transform(train_df["text"])

## note that we're NOT using .fit_transform() here. Using just .transform() makes sure
# that the tokens in the train vectors are the only ones mapped to the test vectors - 
# i.e. that the train and test vectors use the same set of tokens.
test_vectors = count_vectorizer.transform(test_df["text"])

### Our model

As we mentioned above, we think the words contained in each tweet are a good indicator of whether they're about a real disaster or not. The presence of particular word (or set of words) in a tweet might link directly to whether or not that tweet is real.

What we're assuming here is a _linear_ connection. So let's build a linear model and see!

In [ ]:
## Our vectors are really big, so we want to push our model's weights
## toward 0 without completely discounting different words - ridge regression 
## is a good way to do this.
clf = linear_model.RidgeClassifier()

Let's test our model and see how well it does on the training data. For this we'll use `cross-validation` - where we train on a portion of the known data, then validate it with the rest. If we do this several times (with different portions) we can get a good idea for how a particular model or method performs.

The metric for this competition is F1, so let's use that here.

In [ ]:
scores = model_selection.cross_val_score(clf, train_vectors, train_df["target"], cv=3, scoring="f1")
scores

The above scores aren't terrible! It looks like our assumption will score roughly 0.65 on the leaderboard. There are lots of ways to potentially improve on this (TFIDF, LSA, LSTM / RNNs, the list is long!) - give any of them a shot!

In the meantime, let's do predictions on our training set and build a submission for the competition.

In [ ]:
clf.fit(train_vectors, train_df["target"])

In [ ]:
sample_submission = pd.read_csv(os.path.join(WORK_DIR, "sample_submission.csv"))

In [ ]:
sample_submission["target"] = clf.predict(test_vectors)

In [ ]:
sample_submission.head()

In [ ]:
sample_submission.to_csv(os.path.join(WORK_DIR, "submission.csv"), index=False)

Now, in the viewer, you can submit the above file to the competition! Good luck!

## 제출 파일 내려받기

`submission.csv` 가 `/content` 에 생성되었습니다. 좌측 파일창에서 받거나 아래 셀로 바로 내려받으세요.

In [ ]:
try:
    from google.colab import files
    files.download(os.path.join(WORK_DIR, "submission.csv"))
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", os.path.join(WORK_DIR, "submission.csv"))

## 🏁 제출하기

위에서 생성된 `submission.csv` 를 아래 페이지에서 업로드해 제출하세요:

👉 **[NLP Getting Started 제출 페이지](https://www.kaggle.com/c/nlp-getting-started/submit)**

- 대회 홈: https://www.kaggle.com/c/nlp-getting-started
> 제출하려면 해당 대회에 Join(규칙 동의)되어 있어야 합니다.